# 📊 대중적 관심도 데이터 수집 가이드

## 크롤링 & API 활용 방법

---

## 📋 개요

이 노트북에서는 e스포츠와 전통 스포츠의 **대중적 관심도**를 측정할 수 있는 다양한 데이터 소스에서 데이터를 수집하는 방법을 다룹니다.

### 📌 수집 가능한 데이터 소스

| 플랫폼 | 수집 방법 | 측정 지표 | 난이도 |
|--------|----------|----------|--------|
| **Google Trends** | pytrends (API) | 검색량 추이 | ⭐ 쉬움 |
| **YouTube** | YouTube Data API | 조회수, 구독자 | ⭐⭐ 보통 |
| **Twitch** | Twitch API | 시청자 수, 채널 | ⭐⭐ 보통 |
| **Reddit** | PRAW (API) | 게시물, 댓글 수 | ⭐⭐ 보통 |
| **Twitter/X** | 크롤링/API | 언급량, 해시태그 | ⭐⭐⭐ 어려움 |
| **뉴스** | 크롤링 | 기사 수, 헤드라인 | ⭐⭐ 보통 |
| **Wikipedia** | API | 페이지 조회수 | ⭐ 쉬움 |

---

## ⚠️ 크롤링 전 주의사항

### 1. 법적/윤리적 고려사항

```
✅ 해야 할 것:
- robots.txt 확인 (예: https://www.youtube.com/robots.txt)
- 서버에 과도한 부하를 주지 않기 (요청 간 딜레이)
- 공식 API가 있으면 API 우선 사용
- 개인정보 수집 금지
- 학술/교육 목적으로만 사용

❌ 하지 말 것:
- 로그인이 필요한 페이지 무단 접근
- 수집한 데이터 상업적 이용
- 짧은 시간에 대량 요청 (DDoS로 간주될 수 있음)
```

### 2. API vs 크롤링

| 방법 | 장점 | 단점 |
|------|------|------|
| **API** | 안정적, 합법적, 구조화된 데이터 | API 키 필요, 요청 제한 |
| **크롤링** | 유연함, API 없는 사이트 가능 | 사이트 변경 시 깨짐, 법적 위험 |

---

## 1️⃣ 필수 라이브러리 설치

In [ ]:
# ============================================
# 필수 라이브러리 설치
# ============================================

# 기본 크롤링
!pip install requests beautifulsoup4 lxml

# Google Trends
!pip install pytrends

# Reddit API
!pip install praw

# YouTube API (Google API Client)
!pip install google-api-python-client

# 동적 페이지 크롤링 (JavaScript 렌더링)
!pip install selenium webdriver-manager

# 데이터 처리
!pip install pandas numpy matplotlib seaborn

print('✅ 라이브러리 설치 완료!')

In [ ]:
# ============================================
# 라이브러리 임포트
# ============================================
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('✅ 라이브러리 로드 완료!')

---

## 2️⃣ Google Trends - 검색량 추이 (가장 쉬움 ⭐)

### 📌 특징
- **API 키 불필요** - pytrends 라이브러리로 바로 사용
- 검색어별 상대적 관심도 (0-100 스케일)
- 시간대별, 지역별 비교 가능
- e스포츠 vs 전통 스포츠 검색량 비교에 최적

In [ ]:
# ============================================
# Google Trends 데이터 수집
# ============================================
from pytrends.request import TrendReq

# pytrends 객체 생성
pytrends = TrendReq(hl='ko', tz=540)  # 한국어, 한국 시간대

# 비교할 검색어 설정 (최대 5개)
keywords = ['League of Legends', 'FIFA', 'NFL', 'Esports', 'World Cup']

# 검색 기간 설정 (최근 5년)
pytrends.build_payload(
    kw_list=keywords,
    cat=0,  # 카테고리 (0: 전체)
    timeframe='today 5-y',  # 최근 5년
    geo='',  # 전세계 (특정 국가: 'KR', 'US' 등)
    gprop=''  # 검색 유형 (웹검색)
)

# 시간별 관심도 데이터 가져오기
trends_data = pytrends.interest_over_time()

print('✅ Google Trends 데이터 수집 완료!')
print(f'   - 기간: {trends_data.index.min()} ~ {trends_data.index.max()}')
print(f'   - 데이터 포인트: {len(trends_data)}개')
display(trends_data.head(10))

In [ ]:
# ============================================
# Google Trends 시각화
# ============================================
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 1. 시간별 추이
for keyword in keywords:
    if keyword in trends_data.columns:
        axes[0].plot(trends_data.index, trends_data[keyword], label=keyword, linewidth=2)

axes[0].set_xlabel('날짜')
axes[0].set_ylabel('검색 관심도 (0-100)')
axes[0].set_title('📈 Google Trends: e스포츠 vs 전통 스포츠 검색량 추이', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# 2. 평균 관심도 비교
avg_interest = trends_data[keywords].mean().sort_values(ascending=True)
colors = ['#9B59B6' if 'League' in k or 'Esports' in k else '#27AE60' for k in avg_interest.index]
axes[1].barh(avg_interest.index, avg_interest.values, color=colors)
axes[1].set_xlabel('평균 검색 관심도')
axes[1].set_title('📊 평균 검색 관심도 비교', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('google_trends_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================
# 국가별 관심도 비교
# ============================================

# 국가별 관심도
interest_by_region = pytrends.interest_by_region(resolution='COUNTRY', inc_low_vol=True, inc_geo_code=False)

# 상위 20개 국가
top_countries = interest_by_region.nlargest(20, 'League of Legends')

print('\n🌍 국가별 관심도 (상위 20개국):')
display(top_countries)

In [ ]:
# ============================================
# 관련 검색어 분석
# ============================================

# 관련 검색어 가져오기
related_queries = pytrends.related_queries()

print('\n🔍 "League of Legends" 관련 인기 검색어:')
if 'League of Legends' in related_queries:
    display(related_queries['League of Legends']['top'].head(10))

In [ ]:
# ============================================
# 한국 시장 집중 분석
# ============================================

# 한국에서의 검색량
pytrends.build_payload(
    kw_list=['롤', '축구', 'K리그', 'LCK', '프리미어리그'],
    timeframe='today 12-m',
    geo='KR'
)

korea_trends = pytrends.interest_over_time()

print('\n🇰🇷 한국 검색량 데이터:')
display(korea_trends.head())

# 시각화
fig, ax = plt.subplots(figsize=(12, 6))
for col in korea_trends.columns:
    if col != 'isPartial':
        ax.plot(korea_trends.index, korea_trends[col], label=col, linewidth=2)

ax.set_title('🇰🇷 한국 검색량 추이 (최근 1년)', fontsize=14, fontweight='bold')
ax.set_xlabel('날짜')
ax.set_ylabel('검색 관심도')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 3️⃣ YouTube Data API - 영상 조회수/구독자 (⭐⭐)

### 📌 API 키 발급 방법
1. [Google Cloud Console](https://console.cloud.google.com/) 접속
2. 새 프로젝트 생성
3. "YouTube Data API v3" 활성화
4. 사용자 인증 정보 → API 키 생성

### 📌 무료 할당량
- 일일 10,000 유닛 (검색 1회 = 100유닛, 채널 정보 1회 = 1유닛)

In [ ]:
# ============================================
# YouTube API 설정
# ============================================
from googleapiclient.discovery import build

# ⚠️ 본인의 API 키로 교체하세요!
YOUTUBE_API_KEY = 'YOUR_YOUTUBE_API_KEY_HERE'

# YouTube API 객체 생성
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

print('✅ YouTube API 설정 완료!')
print('⚠️ API 키를 본인의 키로 교체해야 작동합니다.')

In [ ]:
# ============================================
# 채널 정보 수집 함수
# ============================================

def get_channel_stats(channel_id):
    """채널 ID로 채널 통계 가져오기"""
    try:
        request = youtube.channels().list(
            part='snippet,statistics',
            id=channel_id
        )
        response = request.execute()
        
        if response['items']:
            item = response['items'][0]
            return {
                'channel_name': item['snippet']['title'],
                'subscribers': int(item['statistics'].get('subscriberCount', 0)),
                'views': int(item['statistics'].get('viewCount', 0)),
                'videos': int(item['statistics'].get('videoCount', 0))
            }
    except Exception as e:
        print(f'Error: {e}')
    return None

def search_channels(query, max_results=10):
    """검색어로 채널 검색"""
    try:
        request = youtube.search().list(
            part='snippet',
            q=query,
            type='channel',
            maxResults=max_results
        )
        response = request.execute()
        
        channels = []
        for item in response['items']:
            channels.append({
                'channel_id': item['snippet']['channelId'],
                'channel_name': item['snippet']['title'],
                'description': item['snippet']['description'][:100]
            })
        return channels
    except Exception as e:
        print(f'Error: {e}')
    return []

print('✅ 함수 정의 완료!')

In [ ]:
# ============================================
# 주요 e스포츠/스포츠 채널 비교
# ============================================

# 주요 채널 ID (예시 - 실제 채널 ID로 교체)
channel_ids = {
    # e스포츠
    'LoL Esports': 'UCvqRdlKsE5Q8mf8YXbdIJLw',
    'Riot Games': 'UC2t5bjwHdUX4vM2g8TRDq5g',
    'T1': 'UCaYLBJfw6d8XqmNlL30oc6A',
    
    # 전통 스포츠
    'NBA': 'UCWOF2HFiGMlFsvKm1vjXBYg',
    'NFL': 'UCDVYQ4Zhbm3S2dlz7P1GBDg',
    'UEFA Champions League': 'UCTwC-gDKiHoHrC5FPz0tCxQ'
}

# 데이터 수집 (API 키 설정 후 실행)
'''
channel_stats = []
for name, channel_id in channel_ids.items():
    stats = get_channel_stats(channel_id)
    if stats:
        stats['category'] = 'e스포츠' if name in ['LoL Esports', 'Riot Games', 'T1'] else '전통 스포츠'
        channel_stats.append(stats)
    time.sleep(0.5)  # API 요청 간 딜레이

youtube_df = pd.DataFrame(channel_stats)
display(youtube_df)
'''

print('⚠️ API 키 설정 후 위 코드 블록의 주석을 해제하고 실행하세요.')

In [ ]:
# ============================================
# 샘플 데이터로 시각화 (API 없이 테스트용)
# ============================================

# 예시 데이터
youtube_sample = pd.DataFrame({
    'channel_name': ['LoL Esports', 'Riot Games', 'T1', 'NBA', 'NFL', 'UEFA'],
    'subscribers': [4200000, 2100000, 3500000, 21000000, 11000000, 8500000],
    'views': [2500000000, 1800000000, 1200000000, 15000000000, 8000000000, 5500000000],
    'videos': [8500, 4200, 2800, 65000, 45000, 12000],
    'category': ['e스포츠', 'e스포츠', 'e스포츠', '전통 스포츠', '전통 스포츠', '전통 스포츠']
})

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ['#9B59B6' if c == 'e스포츠' else '#27AE60' for c in youtube_sample['category']]

# 구독자 수
axes[0].barh(youtube_sample['channel_name'], youtube_sample['subscribers']/1e6, color=colors)
axes[0].set_xlabel('구독자 수 (백만)')
axes[0].set_title('📺 YouTube 구독자 수', fontweight='bold')

# 총 조회수
axes[1].barh(youtube_sample['channel_name'], youtube_sample['views']/1e9, color=colors)
axes[1].set_xlabel('총 조회수 (10억)')
axes[1].set_title('👁️ 총 조회수', fontweight='bold')

# 영상 수
axes[2].barh(youtube_sample['channel_name'], youtube_sample['videos'], color=colors)
axes[2].set_xlabel('영상 수')
axes[2].set_title('🎬 업로드 영상 수', fontweight='bold')

plt.tight_layout()
plt.show()

---

## 4️⃣ Reddit API (PRAW) - 커뮤니티 활동량 (⭐⭐)

### 📌 API 키 발급 방법
1. [Reddit Apps](https://www.reddit.com/prefs/apps) 접속 (로그인 필요)
2. "Create App" 클릭
3. "script" 타입 선택
4. client_id, client_secret 저장

In [ ]:
# ============================================
# Reddit API 설정
# ============================================
import praw

# ⚠️ 본인의 Reddit API 정보로 교체하세요!
reddit = praw.Reddit(
    client_id='YOUR_CLIENT_ID',
    client_secret='YOUR_CLIENT_SECRET',
    user_agent='esports_analysis_bot/1.0'
)

print('✅ Reddit API 설정 완료!')
print('⚠️ API 정보를 본인의 것으로 교체해야 작동합니다.')

In [ ]:
# ============================================
# 서브레딧 통계 수집 함수
# ============================================

def get_subreddit_stats(subreddit_name):
    """서브레딧 통계 가져오기"""
    try:
        subreddit = reddit.subreddit(subreddit_name)
        return {
            'name': subreddit.display_name,
            'title': subreddit.title,
            'subscribers': subreddit.subscribers,
            'active_users': subreddit.accounts_active,
            'created_utc': datetime.fromtimestamp(subreddit.created_utc)
        }
    except Exception as e:
        print(f'Error for r/{subreddit_name}: {e}')
    return None

def get_recent_posts(subreddit_name, limit=100):
    """최근 게시물 가져오기"""
    try:
        subreddit = reddit.subreddit(subreddit_name)
        posts = []
        for post in subreddit.new(limit=limit):
            posts.append({
                'title': post.title,
                'score': post.score,
                'num_comments': post.num_comments,
                'created_utc': datetime.fromtimestamp(post.created_utc),
                'upvote_ratio': post.upvote_ratio
            })
        return posts
    except Exception as e:
        print(f'Error: {e}')
    return []

print('✅ 함수 정의 완료!')

In [ ]:
# ============================================
# 주요 서브레딧 비교
# ============================================

# 비교할 서브레딧 목록
subreddits = {
    'e스포츠': ['leagueoflegends', 'GlobalOffensive', 'DotA2', 'VALORANT', 'esports'],
    '전통 스포츠': ['soccer', 'nfl', 'nba', 'baseball', 'hockey']
}

# 데이터 수집 (API 설정 후 실행)
'''
all_stats = []
for category, subs in subreddits.items():
    for sub in subs:
        stats = get_subreddit_stats(sub)
        if stats:
            stats['category'] = category
            all_stats.append(stats)
        time.sleep(1)  # API 요청 간 딜레이

reddit_df = pd.DataFrame(all_stats)
display(reddit_df.sort_values('subscribers', ascending=False))
'''

print('⚠️ API 설정 후 위 코드 블록의 주석을 해제하고 실행하세요.')

In [ ]:
# ============================================
# 샘플 데이터로 시각화 (API 없이 테스트용)
# ============================================

reddit_sample = pd.DataFrame({
    'name': ['leagueoflegends', 'GlobalOffensive', 'VALORANT', 'soccer', 'nfl', 'nba'],
    'subscribers': [6800000, 2100000, 2300000, 4500000, 3200000, 8500000],
    'active_users': [15000, 8000, 12000, 25000, 18000, 35000],
    'category': ['e스포츠', 'e스포츠', 'e스포츠', '전통 스포츠', '전통 스포츠', '전통 스포츠']
})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#9B59B6' if c == 'e스포츠' else '#27AE60' for c in reddit_sample['category']]

# 구독자 수
reddit_sorted = reddit_sample.sort_values('subscribers', ascending=True)
colors_sorted = ['#9B59B6' if c == 'e스포츠' else '#27AE60' for c in reddit_sorted['category']]

axes[0].barh(reddit_sorted['name'], reddit_sorted['subscribers']/1e6, color=colors_sorted)
axes[0].set_xlabel('구독자 수 (백만)')
axes[0].set_title('📱 Reddit 서브레딧 구독자 수', fontweight='bold')

# 활성 사용자
reddit_active = reddit_sample.sort_values('active_users', ascending=True)
colors_active = ['#9B59B6' if c == 'e스포츠' else '#27AE60' for c in reddit_active['category']]

axes[1].barh(reddit_active['name'], reddit_active['active_users']/1000, color=colors_active)
axes[1].set_xlabel('현재 활성 사용자 (천명)')
axes[1].set_title('🟢 현재 활성 사용자', fontweight='bold')

plt.tight_layout()
plt.show()

---

## 5️⃣ Wikipedia API - 페이지 조회수 (⭐ 쉬움)

### 📌 특징
- **API 키 불필요**
- 일별/월별 페이지 조회수 제공
- 대중적 관심도의 객관적 지표

In [ ]:
# ============================================
# Wikipedia 페이지 조회수 수집
# ============================================

def get_wikipedia_pageviews(article_title, start_date, end_date, language='en'):
    """
    Wikipedia 페이지 조회수 가져오기
    
    Parameters:
    - article_title: Wikipedia 문서 제목
    - start_date: 시작일 (YYYYMMDD)
    - end_date: 종료일 (YYYYMMDD)
    - language: 언어 코드 (en, ko 등)
    """
    # 공백을 언더스코어로 변환
    article_title = article_title.replace(' ', '_')
    
    url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/{language}.wikipedia/all-access/all-agents/{article_title}/daily/{start_date}/{end_date}"
    
    headers = {
        'User-Agent': 'EsportsAnalysis/1.0 (educational purpose)'
    }
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json()
            df = pd.DataFrame(data['items'])
            df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y%m%d00')
            return df[['timestamp', 'views']]
        else:
            print(f'Error {response.status_code}: {response.text}')
    except Exception as e:
        print(f'Error: {e}')
    return None

print('✅ 함수 정의 완료!')

In [ ]:
# ============================================
# 주요 페이지 조회수 비교
# ============================================

# 비교할 Wikipedia 문서
articles = {
    'League of Legends': 'League_of_Legends',
    'Esports': 'Esports',
    'FIFA World Cup': 'FIFA_World_Cup',
    'Super Bowl': 'Super_Bowl',
    'NBA': 'National_Basketball_Association'
}

# 최근 1년 데이터 수집
start_date = '20240101'
end_date = '20241231'

wiki_data = {}
for name, title in articles.items():
    print(f'수집 중: {name}...')
    df = get_wikipedia_pageviews(title, start_date, end_date)
    if df is not None:
        wiki_data[name] = df
    time.sleep(0.5)  # 요청 간 딜레이

print('\n✅ Wikipedia 데이터 수집 완료!')

In [ ]:
# ============================================
# Wikipedia 조회수 시각화
# ============================================

if wiki_data:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # 1. 시간별 추이
    for name, df in wiki_data.items():
        # 주간 평균으로 스무딩
        df_weekly = df.set_index('timestamp').resample('W').mean()
        axes[0].plot(df_weekly.index, df_weekly['views'], label=name, linewidth=2)
    
    axes[0].set_xlabel('날짜')
    axes[0].set_ylabel('일일 조회수')
    axes[0].set_title('📖 Wikipedia 페이지 조회수 추이 (주간 평균)', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. 총 조회수 비교
    total_views = {name: df['views'].sum() for name, df in wiki_data.items()}
    sorted_views = dict(sorted(total_views.items(), key=lambda x: x[1]))
    
    colors = ['#9B59B6' if 'League' in k or 'Esports' in k else '#27AE60' for k in sorted_views.keys()]
    axes[1].barh(list(sorted_views.keys()), [v/1e6 for v in sorted_views.values()], color=colors)
    axes[1].set_xlabel('총 조회수 (백만)')
    axes[1].set_title('📊 연간 총 조회수 비교', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('wikipedia_pageviews.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('데이터가 없습니다.')

---

## 6️⃣ 웹 크롤링 기본 - BeautifulSoup (⭐⭐)

### 📌 사용 사례
- 뉴스 기사 수집
- 순위 정보 수집
- 통계 페이지 크롤링

In [ ]:
# ============================================
# 기본 크롤링 함수
# ============================================

def fetch_page(url):
    """웹 페이지 가져오기"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        return BeautifulSoup(response.text, 'lxml')
    except Exception as e:
        print(f'Error fetching {url}: {e}')
    return None

print('✅ 크롤링 함수 정의 완료!')

In [ ]:
# ============================================
# 예시: Esports Charts 크롤링 (시청자 통계)
# ============================================

def crawl_esports_charts():
    """
    Esports Charts에서 인기 게임 순위 크롤링
    ⚠️ 실제 사용 전 robots.txt 확인 필요
    """
    url = 'https://escharts.com/top-games'
    soup = fetch_page(url)
    
    if soup:
        # 실제 HTML 구조에 맞게 수정 필요
        # 아래는 예시 코드입니다
        games = []
        game_items = soup.find_all('div', class_='game-item')  # 실제 클래스명 확인 필요
        
        for item in game_items:
            name = item.find('span', class_='game-name')
            viewers = item.find('span', class_='viewers')
            if name and viewers:
                games.append({
                    'game': name.text.strip(),
                    'viewers': viewers.text.strip()
                })
        
        return games
    return []

# 실행 (실제 HTML 구조 확인 후 수정 필요)
# games_data = crawl_esports_charts()
print('⚠️ 실제 크롤링 전 대상 사이트의 robots.txt와 이용약관을 확인하세요.')

In [ ]:
# ============================================
# 예시: 네이버 뉴스 검색 크롤링
# ============================================

def search_naver_news(query, pages=3):
    """
    네이버 뉴스 검색 결과 크롤링
    
    Parameters:
    - query: 검색어
    - pages: 수집할 페이지 수
    """
    all_articles = []
    
    for page in range(1, pages + 1):
        start = (page - 1) * 10 + 1
        url = f'https://search.naver.com/search.naver?where=news&query={query}&start={start}'
        
        soup = fetch_page(url)
        if not soup:
            continue
        
        # 뉴스 아이템 찾기
        news_items = soup.find_all('div', class_='news_area')
        
        for item in news_items:
            title_tag = item.find('a', class_='news_tit')
            if title_tag:
                all_articles.append({
                    'title': title_tag.get('title', ''),
                    'url': title_tag.get('href', ''),
                    'query': query
                })
        
        time.sleep(1)  # 요청 간 딜레이
    
    return all_articles

# 테스트
print('검색어별 뉴스 기사 수집 중...')
queries = ['e스포츠', '프로야구', 'K리그']

news_counts = {}
for query in queries:
    articles = search_naver_news(query, pages=2)
    news_counts[query] = len(articles)
    print(f'  - "{query}": {len(articles)}개 기사')

print('\n✅ 뉴스 검색 완료!')

---

## 7️⃣ Selenium - 동적 페이지 크롤링 (⭐⭐⭐)

### 📌 사용 사례
- JavaScript로 렌더링되는 페이지
- 무한 스크롤 페이지
- 로그인 후 접근 가능한 페이지

In [ ]:
# ============================================
# Selenium 설정
# ============================================
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def create_driver(headless=True):
    """Chrome WebDriver 생성"""
    options = Options()
    if headless:
        options.add_argument('--headless')  # 브라우저 창 숨김
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver

print('✅ Selenium 설정 완료!')

In [ ]:
# ============================================
# 예시: Twitch 디렉토리 크롤링
# ============================================

def crawl_twitch_directory():
    """
    Twitch 게임 디렉토리에서 인기 게임 크롤링
    ⚠️ Twitch API 사용을 권장합니다
    """
    driver = create_driver(headless=True)
    
    try:
        url = 'https://www.twitch.tv/directory'
        driver.get(url)
        
        # 페이지 로딩 대기
        time.sleep(5)
        
        # 게임 카드 찾기 (실제 클래스명 확인 필요)
        games = []
        game_cards = driver.find_elements(By.CSS_SELECTOR, '[data-a-target="card-0"]')
        
        for card in game_cards[:20]:  # 상위 20개
            try:
                name = card.find_element(By.CSS_SELECTOR, 'h2').text
                viewers = card.find_element(By.CSS_SELECTOR, 'p').text
                games.append({'game': name, 'viewers': viewers})
            except:
                continue
        
        return games
    
    finally:
        driver.quit()

# 실행 (Selenium 설치 후)
# twitch_games = crawl_twitch_directory()
print('⚠️ Twitch는 공식 API 사용을 권장합니다.')
print('   API 문서: https://dev.twitch.tv/docs/api/')

---

## 8️⃣ Twitch API - 스트리밍 데이터 (⭐⭐)

### 📌 API 키 발급 방법
1. [Twitch Developer Console](https://dev.twitch.tv/console) 접속
2. "Register Your Application" 클릭
3. Client ID와 Client Secret 저장

In [ ]:
# ============================================
# Twitch API 설정
# ============================================

# ⚠️ 본인의 Twitch API 정보로 교체하세요!
TWITCH_CLIENT_ID = 'YOUR_CLIENT_ID'
TWITCH_CLIENT_SECRET = 'YOUR_CLIENT_SECRET'

def get_twitch_token():
    """Twitch OAuth 토큰 발급"""
    url = 'https://id.twitch.tv/oauth2/token'
    params = {
        'client_id': TWITCH_CLIENT_ID,
        'client_secret': TWITCH_CLIENT_SECRET,
        'grant_type': 'client_credentials'
    }
    response = requests.post(url, params=params)
    return response.json().get('access_token')

def get_top_games(token, limit=20):
    """인기 게임 목록 가져오기"""
    url = 'https://api.twitch.tv/helix/games/top'
    headers = {
        'Client-ID': TWITCH_CLIENT_ID,
        'Authorization': f'Bearer {token}'
    }
    params = {'first': limit}
    
    response = requests.get(url, headers=headers, params=params)
    return response.json().get('data', [])

def get_streams_by_game(token, game_id, limit=20):
    """게임별 현재 스트림 가져오기"""
    url = 'https://api.twitch.tv/helix/streams'
    headers = {
        'Client-ID': TWITCH_CLIENT_ID,
        'Authorization': f'Bearer {token}'
    }
    params = {'game_id': game_id, 'first': limit}
    
    response = requests.get(url, headers=headers, params=params)
    return response.json().get('data', [])

print('✅ Twitch API 함수 정의 완료!')
print('⚠️ API 정보를 본인의 것으로 교체해야 작동합니다.')

In [ ]:
# ============================================
# Twitch 데이터 수집 (API 설정 후 실행)
# ============================================

'''
# 토큰 발급
token = get_twitch_token()

# 인기 게임 목록
top_games = get_top_games(token, limit=20)
print('\n🎮 Twitch 인기 게임 Top 20:')
for i, game in enumerate(top_games, 1):
    print(f"  {i}. {game['name']}")

# 게임별 현재 시청자 수 합계
for game in top_games[:5]:
    streams = get_streams_by_game(token, game['id'], limit=100)
    total_viewers = sum(s['viewer_count'] for s in streams)
    print(f"  {game['name']}: {total_viewers:,}명 시청 중")
    time.sleep(0.5)
'''

print('⚠️ API 설정 후 위 코드 블록의 주석을 해제하고 실행하세요.')

---

## 9️⃣ 데이터 통합 및 저장

In [ ]:
# ============================================
# 수집된 데이터 통합 예시
# ============================================

# 각 소스에서 수집한 데이터를 통합
def create_popularity_index(google_trends, youtube_data, reddit_data, wiki_data):
    """
    다양한 소스의 데이터를 통합하여 인기도 지수 생성
    
    가중치 예시:
    - Google Trends: 30%
    - YouTube: 25%
    - Reddit: 20%
    - Wikipedia: 25%
    """
    # 각 지표를 0-100 스케일로 정규화
    def normalize(series):
        return (series - series.min()) / (series.max() - series.min()) * 100
    
    # 통합 인덱스 계산
    # ... 실제 데이터로 구현
    
    return None

print('✅ 데이터 통합 함수 정의 완료!')

In [ ]:
# ============================================
# 데이터 저장
# ============================================

def save_data(data, filename, format='csv'):
    """
    수집된 데이터를 파일로 저장
    
    Parameters:
    - data: DataFrame 또는 dict
    - filename: 저장할 파일명
    - format: 'csv', 'json', 'excel'
    """
    if isinstance(data, dict):
        data = pd.DataFrame(data)
    
    if format == 'csv':
        data.to_csv(f'{filename}.csv', index=False, encoding='utf-8-sig')
    elif format == 'json':
        data.to_json(f'{filename}.json', orient='records', force_ascii=False)
    elif format == 'excel':
        data.to_excel(f'{filename}.xlsx', index=False)
    
    print(f'✅ 데이터 저장 완료: {filename}.{format}')

# 예시: Google Trends 데이터 저장
# save_data(trends_data, 'google_trends_esports', 'csv')

print('✅ 저장 함수 정의 완료!')

---

## 📋 요약: 데이터 소스별 권장 사항

| 데이터 소스 | 수집 방법 | 난이도 | 권장 용도 |
|------------|----------|--------|----------|
| **Google Trends** | pytrends | ⭐ | 검색량 추이, 지역별 관심도 |
| **YouTube** | YouTube Data API | ⭐⭐ | 영상 조회수, 채널 구독자 |
| **Reddit** | PRAW | ⭐⭐ | 커뮤니티 활동량, 게시물 분석 |
| **Wikipedia** | REST API | ⭐ | 페이지 조회수, 객관적 인기도 |
| **Twitch** | Twitch API | ⭐⭐ | 실시간 시청자, 스트리밍 통계 |
| **뉴스** | 크롤링 | ⭐⭐ | 미디어 노출량, 트렌드 분석 |

### ⚠️ 주의사항
1. **API 우선 사용**: 크롤링보다 API가 안정적이고 합법적
2. **요청 제한 준수**: 각 API의 rate limit 확인
3. **딜레이 추가**: 요청 간 1-2초 대기
4. **robots.txt 확인**: 크롤링 전 반드시 확인
5. **데이터 백업**: 수집한 데이터는 즉시 저장

---

## 📚 참고 자료

### API 문서
- [Google Trends (pytrends)](https://pypi.org/project/pytrends/)
- [YouTube Data API](https://developers.google.com/youtube/v3)
- [Reddit API (PRAW)](https://praw.readthedocs.io/)
- [Twitch API](https://dev.twitch.tv/docs/api/)
- [Wikipedia API](https://www.mediawiki.org/wiki/API:Main_page)

### 크롤링 라이브러리
- [BeautifulSoup](https://beautiful-soup-4.readthedocs.io/)
- [Selenium](https://selenium-python.readthedocs.io/)
- [Scrapy](https://scrapy.org/) (대규모 크롤링용)

---

**작성일**: 2025년 1월  
**용도**: 대중적 관심도 데이터 수집 가이드